In [1]:
import pandas as pd

from db import fetch_rows, fetch_df, fetch_table_rows, fetch_table_df, execute_write

```python
# Reference Functions
fetch_rows(query, params=None)  # Execute any query and return rows as a list of dicts
fetch_table_rows(table_name, limit=None)  # Fetch rows from a table with an optional limit
fetch_table_df(table_name, limit=None)  # Fetch df of any table with an optional limit
execute_write(query, params=None)  # Execute any write query (INSERT, UPDATE, DELETE) with optional parameters. Returns None.
```

In [2]:
# # delete all data from all tables and reset auto-incrementing ids
# # Dangerous reset helper example.
# execute_write("""
# TRUNCATE TABLE 
# questions,
# users,
# subjects,
# topics,
# reviews,
# badges,
# stats,
# settings,
# user_subjects,
# user_badges,
# sync_meta
# RESTART IDENTITY CASCADE;
# """)


In [16]:
# schema info
schema_df = fetch_df("""
SELECT
    conrelid::regclass AS table_name,
    conname AS constraint_name,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE contype IN ('p','u')
ORDER BY table_name;
""")
schema_df.head(2)

,table_name,constraint_name,definition
0,pg_default_acl,pg_default_acl_role_nsp_obj_index,"UNIQUE (defaclrole, defaclnamespace, defaclobj..."
1,pg_default_acl,pg_default_acl_oid_index,PRIMARY KEY (oid)


In [30]:
# pg_stat_user_tables content and table name, row counts
pg_stat_user_tables_df = fetch_table_df("pg_stat_user_tables", limit=None)
pg_stat_user_tables_df.rename(columns={"relname": "table_name", "n_live_tup": "row_count"}, inplace=True)
pg_stat_user_tables_df[["table_name", "row_count"]].sort_values("table_name").reset_index(drop=True)

,table_name,row_count
0,badges,0
1,questions,3738
2,reviews,985
3,settings,105
4,stats,1459
5,subjects,3
6,sync_meta,0
7,topics,23
8,user_badges,14
9,user_subjects,3


In [31]:
# badges table content
badges_df = fetch_table_df("badges", limit=None)
badges_df.head(2)

""


In [20]:
# Questions table content
Questions_table = fetch_df("""
SELECT *
FROM questions
""")
Questions_table.head(2)

,id,topic_id,type,question,answer
0,1,6,math-addition,4 + 3 = ?,7
1,2,6,math-addition,9 + 6 = ?,15


In [32]:
# reviews table content
reviews_df = fetch_table_df("reviews", limit=None)
reviews_df.head(2)

,user_id,question_id,repetition,interval,ease_factor,next_review,last_result,rev_id,last_modified_rev,sync_version,updated_at
0,1,100310,1,1,2.5,1775548168278,good,1775833626151,1775833626151,2,2026-04-10 20:37:05.992
1,1,100321,1,1,2.5,1775548472542,good,1775833626162,1775833626162,2,2026-04-10 20:37:05.992


In [19]:
# Settings table content
Settings_table = fetch_table_df("settings", 150).sort_values(["user_id", "key"]).reset_index(drop=True)
Settings_table.head(2)

,user_id,key,value,updated_at,sync_version
0,0,admin_selected_topic_path_1:2,11,2026-04-10 20:37:53.699,2
1,0,admin_selected_topic_path_1:3,21,2026-04-09 02:25:01.760,1


In [35]:
# stats table content
stats_df = fetch_table_df("stats", 150).sort_values(["user_id", "topic_id", "question_id", "wrong"]).reset_index(drop=True)
stats_df.head(2)

,id,user_id,question_id,topic_id,correct,wrong,practiced_at,updated_at
0,17015,1,100351,2,1,0,2026-04-06 12:24:27.824,2026-04-06 12:24:27.824
1,17016,1,100352,2,1,0,2026-04-06 12:24:59.064,2026-04-06 12:24:59.064


In [36]:
# subjects table content
subjects_df = fetch_table_df("subjects", limit=None)
subjects_df.head(2)

,id,name
0,1,Mathematics
1,2,English


In [37]:
# sync_meta table content
sync_meta_df = fetch_table_df("sync_meta", limit=None)
sync_meta_df.head(2)

""


In [38]:
# topics table content
topics_df = fetch_table_df("topics", limit=None)
topics_df.head(2)

,id,subject_id,parent_topic_id,key,name
0,1,1,NaN,multiplication_tables,Multiplication Tables
1,2,1,1.0,tables_1_5,Tables 1-5


In [40]:
# user_badges table content
user_badges_df = fetch_table_df("user_badges", limit=None)
user_badges_df.head(2)

,user_id,badge_id,unlocked_at,updated_at,sync_version
0,3,topic_master,2026-04-06 04:46:14.036,2026-04-08 11:56:03.808,43
1,3,100_questions,2026-04-09 11:36:59.603,2026-04-09 11:37:22.461,3


In [41]:
# user_subjects table content
user_subjects_df = fetch_table_df("user_subjects", limit=None)
user_subjects_df.head(2)

,user_id,subject_id
0,1,1
1,2,1


In [45]:
# users table content
users_df = fetch_table_df("users", limit=None)
users_df

,id,name,created_at
0,1,Bhavi,2026-04-09 10:52:32.799927
1,2,Madhu,2026-04-09 10:52:32.799927
2,3,Quiz Kid,2026-04-09 10:52:32.799927


In [ ]:
# User names and their registered devices (from settings table)
fetch_df("""
SELECT
  u.name AS user_name,
  d->>'backendKey' AS backend_key,
  d->>'displayName' AS display_name
FROM settings s
JOIN users u
  ON u.id = s.user_id
CROSS JOIN LATERAL jsonb_array_elements(s.value::jsonb) AS d
WHERE s.key LIKE %s
ORDER BY u.name, display_name;
""", ('device_registry_user_%',))


,user_name,backend_key,display_name
0,Bhavi,device_bhavi_phone,bhavi_phone
1,Bhavi,device_bhavi_tab,bhavi_tab
2,Bhavi,device_eshu_s22,eshu_s22
3,Bhavi,device_eshu_tablet,eshu_tablet
4,Bhavi,device_mabhu_phone,mabhu_phone
5,Bhavi,device_mabhu_tab,mabhu_tab
6,Madhu,device_bhavi_phone,bhavi_phone
7,Madhu,device_bhavi_tab,bhavi_tab
8,Madhu,device_eshu_s22,eshu_s22
9,Madhu,device_eshu_tablet,eshu_tablet


In [ ]:
# User names and their active device (from settings table)
fetch_df("""
WITH registry AS (
  SELECT
    s.user_id,
    d->>'backendKey' AS backend_key,
    d->>'displayName' AS display_name
  FROM settings s
  CROSS JOIN LATERAL jsonb_array_elements(s.value::jsonb) AS d
  WHERE s.key LIKE %s
),
active AS (
  SELECT
    s.user_id,
    s.value AS active_backend_key
  FROM settings s
  WHERE s.key LIKE %s
)
SELECT
  u.name AS user_name,
  a.active_backend_key,
  r.display_name
FROM active a
JOIN users u
  ON u.id = a.user_id
LEFT JOIN registry r
  ON r.user_id = a.user_id
 AND r.backend_key = a.active_backend_key
ORDER BY u.name;
""", ('device_registry_user_%', 'active_device_key_user_%'))


,user_name,active_backend_key,display_name
0,Bhavi,device_bhavi_tab,bhavi_tab
1,Madhu,device_mabhu_tab,mabhu_tab
2,Quiz Kid,device_eshu_tablet,eshu_tablet


In [67]:
# User names, subjects, and their selected topic paths (from settings table)
fetch_df("""
WITH RECURSIVE
selected_paths AS (
  SELECT
    regexp_replace(key, '^admin_selected_topic_path_', '') AS user_subject_key,
    value::bigint AS selected_topic_id
  FROM settings
  WHERE key LIKE %s
),
parsed AS (
  SELECT
    split_part(user_subject_key, ':', 1)::bigint AS user_id,
    split_part(user_subject_key, ':', 2)::bigint AS subject_id,
    selected_topic_id
  FROM selected_paths
),
branch AS (
  SELECT
    p.user_id,
    p.subject_id,
    p.selected_topic_id AS topic_id,
    p.selected_topic_id AS root_topic_id,
    0 AS depth
  FROM parsed p
  UNION ALL
  SELECT
    b.user_id,
    b.subject_id,
    t.id AS topic_id,
    b.root_topic_id,
    b.depth + 1 AS depth
  FROM branch b
  JOIN topics t
    ON t.parent_topic_id = b.topic_id
)
SELECT
  u.name AS user_name,
  s.name AS subject_name,
  CASE
    WHEN root.parent_topic_id IS NULL THEN root.name
    ELSE root_parent.name
  END AS selected_root_level1,
  CASE
    WHEN root.parent_topic_id IS NULL THEN NULL
    ELSE root.name
  END AS selected_root_level2,
  t.name AS visible_topic,
  CASE
    WHEN t.parent_topic_id IS NULL THEN 'level1'
    ELSE 'level2'
  END AS visible_level,
  b.depth
FROM branch b
JOIN users u
  ON u.id = b.user_id
JOIN subjects s
  ON s.id = b.subject_id
JOIN topics t
  ON t.id = b.topic_id
LEFT JOIN topics root
  ON root.id = b.root_topic_id
LEFT JOIN topics root_parent
  ON root_parent.id = root.parent_topic_id
ORDER BY u.name, s.name, b.depth, t.name;
""", ('admin_selected_topic_path_%',))


,user_name,subject_name,selected_root_level1,selected_root_level2,visible_topic,visible_level,depth
0,Bhavi,English,Word Spellings,None,Word Spellings,level1,0
1,Bhavi,English,Word Spellings,None,2 Letter Words,level2,1
2,Bhavi,English,Word Spellings,None,3 Letter Words,level2,1
3,Bhavi,English,Word Spellings,None,4 Letter Words,level2,1
4,Bhavi,English,Word Spellings,None,5 Letter Words,level2,1
5,Bhavi,English,Word Spellings,None,6 Letter Words,level2,1
6,Bhavi,English,Word Spellings,None,7 Letter Words,level2,1
7,Bhavi,Science,Science Spellings,None,Science Spellings,level1,0
8,Bhavi,Science,Science Spellings,None,Science Long Words,level2,1
9,Bhavi,Science,Science Spellings,None,Science Short Words,level2,1


In [ ]:
# User names, subjects, and their selected topic in topics page (from settings table)
fetch_df("""
WITH selected_topics AS (
  SELECT
    user_id,
    'level1' AS selected_level,
    jsonb_array_elements_text(value::jsonb)::bigint AS topic_id
  FROM settings
  WHERE key LIKE %s

  UNION ALL

  SELECT
    user_id,
    'level2' AS selected_level,
    jsonb_array_elements_text(value::jsonb)::bigint AS topic_id
  FROM settings
  WHERE key LIKE %s
)
SELECT
  u.name AS user_name,
  s.name AS subject_name,
  CASE
    WHEN t.parent_topic_id IS NULL THEN t.name
    ELSE tp.name
  END AS topic_level1,
  CASE
    WHEN t.parent_topic_id IS NULL THEN NULL
    ELSE t.name
  END AS topic_level2,
  st.selected_level,
  st.topic_id AS selected_topic_id
FROM selected_topics st
JOIN users u
  ON u.id = st.user_id
LEFT JOIN topics t
  ON t.id = st.topic_id
LEFT JOIN topics tp
  ON tp.id = t.parent_topic_id
LEFT JOIN subjects s
  ON s.id = t.subject_id
ORDER BY u.name, s.name, st.selected_level, topic_level1, topic_level2;
""", ('selected_topic_level1_ids%', 'selected_topic_level2_ids%'))


,user_name,subject_name,topic_level1,topic_level2,selected_level,selected_topic_id
0,Bhavi,English,Word Spellings,NaN,level1,11
1,Bhavi,English,Word Spellings,4 Letter Words,level2,14
2,Madhu,English,Word Spellings,3 Letter Words,level2,13
3,Quiz Kid,English,Word Spellings,2 Letter Words,level2,12
4,Quiz Kid,Mathematics,Multiplication Tables,Tables 1-5,level2,2
5,Quiz Kid,Mathematics,Multiplication Tables,Tables 11-15,level2,4


In [ ]:
# User names, subjects, topics and their stats from (joining questions, stats, reviews, and settings tables)
fetch_df("""
WITH RECURSIVE
params AS (
  SELECT %s::bigint AS user_id,
         (EXTRACT(EPOCH FROM NOW()) * 1000)::bigint AS now_ms
),
topic_tree AS (
  SELECT
    id,
    name,
    subject_id,
    parent_topic_id,
    0 AS depth,
    CAST(id AS TEXT) AS sort_path
  FROM topics
  WHERE parent_topic_id IS NULL

  UNION ALL

  SELECT
    t.id,
    t.name,
    t.subject_id,
    t.parent_topic_id,
    tt.depth + 1,
    tt.sort_path || '.' || t.id
  FROM topics t
  JOIN topic_tree tt
    ON t.parent_topic_id = tt.id
),
descendants AS (
  SELECT id AS topic_id, id AS descendant_topic_id
  FROM topics

  UNION ALL

  SELECT d.topic_id, t.id
  FROM descendants d
  JOIN topics t
    ON t.parent_topic_id = d.descendant_topic_id
),
question_stats AS (
  SELECT
    q.id AS question_id,
    q.topic_id,
    q.type AS question_type,
    COALESCE(SUM(s.correct), 0) AS correct,
    COALESCE(SUM(s.wrong), 0) AS wrong,
    COALESCE(SUM(COALESCE(s.correct, 0) + COALESCE(s.wrong, 0)), 0) AS attempts
  FROM questions q
  LEFT JOIN stats s
    ON s.question_id = q.id::text
   AND s.user_id = (SELECT user_id FROM params)
  GROUP BY q.id, q.topic_id, q.type
),
review_snapshot AS (
  SELECT
    CASE
      WHEN r.question_id ~ '^[0-9]+$' THEN r.question_id::bigint
      ELSE NULL
    END AS question_id,
    r.repetition,
    r.next_review,
    r.last_result
  FROM reviews r
  WHERE r.user_id = (SELECT user_id FROM params)
),
question_state AS (
  SELECT
    qs.question_id,
    qs.topic_id,
    qs.question_type,
    qs.correct,
    qs.wrong,
    qs.attempts,
    CASE
      WHEN rs.question_id IS NULL THEN 'unseen'
      WHEN rs.last_result = 'again' THEN 'wrong'
      WHEN rs.next_review IS NOT NULL
       AND rs.next_review <= (SELECT now_ms FROM params) THEN 'due'
      WHEN COALESCE(rs.repetition, 0) < 2 THEN 'in_progress'
      ELSE 'recently_mastered'
    END AS stage
  FROM question_stats qs
  LEFT JOIN review_snapshot rs
    ON rs.question_id = qs.question_id
),
topic_rollup AS (
  SELECT
    tt.id AS topic_id,
    tt.name AS topic_name,
    tt.subject_id,
    tt.parent_topic_id,
    tt.depth,
    tt.sort_path,
    COALESCE(COUNT(qs.question_id), 0) AS total_available,
    COALESCE(SUM(qs.attempts), 0) AS total_completed,
    COALESCE(SUM(qs.correct), 0) AS correct,
    COALESCE(SUM(qs.wrong), 0) AS wrong,
    COALESCE(COUNT(*) FILTER (WHERE qs.stage = 'unseen'), 0) AS unseen_cards,
    COALESCE(COUNT(*) FILTER (WHERE qs.stage = 'wrong'), 0) AS wrong_cards,
    COALESCE(COUNT(*) FILTER (WHERE qs.stage = 'due'), 0) AS due_cards,
    COALESCE(COUNT(*) FILTER (WHERE qs.stage = 'in_progress'), 0) AS in_progress_cards,
    COALESCE(COUNT(*) FILTER (WHERE qs.stage = 'recently_mastered'), 0) AS recently_mastered_cards,
    ROUND(
      100.0 * COALESCE(SUM(qs.correct), 0) / NULLIF(COALESCE(SUM(qs.attempts), 0), 0),
      1
    ) AS accuracy_pct,
    ROUND(
      100.0 * COALESCE(SUM(qs.attempts), 0) / NULLIF(COUNT(qs.question_id), 0),
      1
    ) AS completion_pct,
    ROUND(
      1.0 * COALESCE(SUM(qs.attempts), 0) / NULLIF(COUNT(qs.question_id), 0),
      2
    ) AS avg_attempts_per_card
  FROM topic_tree tt
  LEFT JOIN descendants d
    ON d.topic_id = tt.id
  LEFT JOIN question_state qs
    ON qs.topic_id = d.descendant_topic_id
  GROUP BY
    tt.id,
    tt.name,
    tt.subject_id,
    tt.parent_topic_id,
    tt.depth,
    tt.sort_path
),
topic_type_counts AS (
  SELECT
    tt.id AS topic_id,
    qs.question_type,
    COUNT(*) AS count
  FROM topic_tree tt
  LEFT JOIN descendants d
    ON d.topic_id = tt.id
  LEFT JOIN question_state qs
    ON qs.topic_id = d.descendant_topic_id
  GROUP BY tt.id, qs.question_type
),
topic_type_rollup AS (
  SELECT
    topic_id,
    COALESCE(
      jsonb_object_agg(question_type, count)
        FILTER (WHERE question_type IS NOT NULL),
      '{}'::jsonb
    ) AS question_type_counts
  FROM topic_type_counts
  GROUP BY topic_id
)
SELECT
  u.name AS user_name,
  s.name AS subject_name,
  CASE
    WHEN tr.parent_topic_id IS NULL THEN tr.topic_name
    ELSE tp.name
  END AS topic_level1,
  CASE
    WHEN tr.parent_topic_id IS NULL THEN NULL
    ELSE tr.topic_name
  END AS topic_level2,
  tr.depth,
  tr.total_available,
  tr.total_completed,
  tr.correct,
  tr.wrong,
  tr.accuracy_pct,
  tr.completion_pct,
  tr.avg_attempts_per_card,
  tr.unseen_cards,
  tr.wrong_cards,
  tr.due_cards,
  tr.in_progress_cards,
  tr.recently_mastered_cards,
  COALESCE(ttc.question_type_counts, '{}'::jsonb) AS question_type_counts
FROM topic_rollup tr
JOIN users u
  ON u.id = (SELECT user_id FROM params)
JOIN subjects s
  ON s.id = tr.subject_id
LEFT JOIN topics tp
  ON tp.id = tr.parent_topic_id
LEFT JOIN topic_type_rollup ttc
  ON ttc.topic_id = tr.topic_id
ORDER BY u.name, s.name, topic_level1, topic_level2 NULLS FIRST;
""", (1,))


,user_name,subject_name,topic_level1,topic_level2,depth,total_available,total_completed,correct,wrong,accuracy_pct,completion_pct,avg_attempts_per_card,unseen_cards,wrong_cards,due_cards,in_progress_cards,recently_mastered_cards,question_type_counts
0,Bhavi,English,Jumbled Words,NaN,0,4,4,4,0,100.0,100.0,1.00,0,0,4,0,0,{'jumble-word': 4}
1,Bhavi,English,Jumbled Words,3 Letter Jumble,1,2,2,2,0,100.0,100.0,1.00,0,0,2,0,0,{'jumble-word': 2}
2,Bhavi,English,Jumbled Words,5 Letter Jumble,1,2,2,2,0,100.0,100.0,1.00,0,0,2,0,0,{'jumble-word': 2}
3,Bhavi,English,Word Spellings,NaN,0,953,352,333,19,94.6,36.9,0.37,623,3,135,189,3,{'english-spell-bee': 953}
4,Bhavi,English,Word Spellings,2 Letter Words,1,27,21,19,2,90.5,77.8,0.78,7,1,19,0,0,{'english-spell-bee': 27}
5,Bhavi,English,Word Spellings,3 Letter Words,1,126,64,62,2,96.9,50.8,0.51,65,0,35,25,1,{'english-spell-bee': 126}
6,Bhavi,English,Word Spellings,4 Letter Words,1,328,119,115,4,96.6,36.3,0.36,214,0,38,75,1,{'english-spell-bee': 328}
7,Bhavi,English,Word Spellings,5 Letter Words,1,241,71,66,5,93.0,29.5,0.29,177,1,27,35,1,{'english-spell-bee': 241}
8,Bhavi,English,Word Spellings,6 Letter Words,1,151,58,54,4,93.1,38.4,0.38,98,0,11,42,0,{'english-spell-bee': 151}
9,Bhavi,English,Word Spellings,7 Letter Words,1,80,19,17,2,89.5,23.8,0.24,62,1,5,12,0,{'english-spell-bee': 80}


In [65]:
# sort questions according to no of attempts from (joining questions, stats, reviews, and settings tables)
fetch_df("""
SELECT
  q.id AS question_id,
  u.name AS user_name,
  s.name AS subject_name,
  CASE
    WHEN t.parent_topic_id IS NULL THEN t.name
    ELSE tp.name
  END AS topic_level1,
  CASE
    WHEN t.parent_topic_id IS NULL THEN NULL
    ELSE t.name
  END AS topic_level2,
  q.type AS question_type,
  q.question,
  q.answer,
  COALESCE(SUM(COALESCE(st.correct, 0) + COALESCE(st.wrong, 0)), 0) AS attempts,
  COALESCE(SUM(st.correct), 0) AS correct,
  COALESCE(SUM(st.wrong), 0) AS wrong,
  CASE
    WHEN COALESCE(SUM(COALESCE(st.correct, 0) + COALESCE(st.wrong, 0)), 0) = 0 THEN 0
    ELSE ROUND(
      100.0 * COALESCE(SUM(st.correct), 0) /
      COALESCE(SUM(COALESCE(st.correct, 0) + COALESCE(st.wrong, 0)), 0),
      1
    )
  END AS accuracy_pct,
  MAX(st.practiced_at) AS last_practiced_at
FROM questions q
LEFT JOIN stats st
  ON st.question_id = q.id::text
LEFT JOIN topics t
  ON t.id = q.topic_id
LEFT JOIN topics tp
  ON tp.id = t.parent_topic_id
LEFT JOIN subjects s
  ON s.id = t.subject_id
LEFT JOIN users u
  ON u.id = st.user_id
GROUP BY
  q.id, u.name, s.name, t.parent_topic_id, t.name, tp.name,
  q.type, q.question, q.answer
ORDER BY attempts DESC, last_practiced_at DESC NULLS LAST, question_id ASC;
""")


,question_id,user_name,subject_name,topic_level1,topic_level2,question_type,question,answer,attempts,correct,wrong,accuracy_pct,last_practiced_at
0,100148,Madhu,Mathematics,Multiplication Tables,Tables 6-10,math-tables,10 x 8,80,12,12,0,100.0,2026-04-07 15:42:36.128
1,100428,Madhu,English,Word Spellings,4 Letter Words,english-spell-bee,They are my friends.,they,12,12,0,100.0,2026-04-07 15:42:36.128
2,18,Madhu,English,Word Spellings,5 Letter Words,english-spell-bee,We plant a seed in soil.,seed,11,11,0,100.0,2026-04-08 21:18:23.141
3,100008,Madhu,Mathematics,Multiplication Tables,Tables 1-5,math-tables,1 x 8,8,11,11,0,100.0,2026-04-08 21:18:23.141
4,15,Quiz Kid,English,Word Spellings,4 Letter Words,english-spell-bee,The fish can swim fast.,swim,11,7,4,63.6,2026-04-08 10:41:44.846
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3956,104662,NaN,Science,Science Spellings,Science Short Words,science-spelling,We played a new board game.,game,0,0,0,0,NaT
3957,104678,NaN,Science,Science Spellings,Science Short Words,science-spelling,French fries are oily.,oily,0,0,0,0,NaT
3958,104699,NaN,Science,Science Spellings,Science Short Words,science-spelling,Let's pack for our trip.,pack,0,0,0,0,NaT
3959,104702,NaN,Science,Science Spellings,Science Long Words,science-spelling,The elderly man needs assistance.,elderly,0,0,0,0,NaT


#### Wrong answered questions (DND)

In [15]:
wrong_ans_ques_df = fetch_df("""
WITH latest_stats AS (
  SELECT DISTINCT ON (s.user_id, s.question_id)
    s.user_id,
    s.question_id,
    s.topic_id,
    s.practiced_at
  FROM stats s
  ORDER BY s.user_id, s.question_id, s.practiced_at DESC
),
wrong_reviews AS (
  SELECT
    r.user_id,
    r.question_id,
    r.last_result,
    r.updated_at
  FROM reviews r
  WHERE r.last_result = 'again'
),
enriched AS (
  SELECT
    wr.user_id,
    u.name AS user_name,
    wr.question_id,
    wr.last_result,
    wr.updated_at,
    COALESCE(ls.topic_id, q.topic_id) AS effective_topic_id,
    q.question AS q_question,
    q.answer   AS q_answer
  FROM wrong_reviews wr
  LEFT JOIN users u
    ON u.id = wr.user_id
  LEFT JOIN latest_stats ls
    ON ls.user_id = wr.user_id
   AND ls.question_id = wr.question_id
  LEFT JOIN questions q
    ON q.id::text = wr.question_id
)
SELECT
  e.user_name,
  s.name AS subject,
  COALESCE(tp.name, t.name) AS topic_level1,
  CASE WHEN tp.id IS NULL THEN NULL ELSE t.name END AS topic_level2,
  e.question_id,
  COALESCE(
    e.q_question,
    CASE
      WHEN e.question_id ~ '^[0-9]+$'
       AND right(e.question_id, 2) = '10'
       AND length(e.question_id) > 2
      THEN
        ((substring(e.question_id from 1 for length(e.question_id)-2))::int)::text
        || ' x 10'
      WHEN e.question_id ~ '^[0-9]+$'
       AND length(e.question_id) > 1
      THEN
        ((substring(e.question_id from 1 for length(e.question_id)-1))::int)::text
        || ' x ' ||
        (right(e.question_id, 1)::int)::text
      ELSE NULL
    END
  ) AS question,
  COALESCE(
    e.q_answer,
    CASE
      WHEN e.question_id ~ '^[0-9]+$'
       AND right(e.question_id, 2) = '10'
       AND length(e.question_id) > 2
      THEN
        (
          (substring(e.question_id from 1 for length(e.question_id)-2))::int
          * 10
        )::text
      WHEN e.question_id ~ '^[0-9]+$'
       AND length(e.question_id) > 1
      THEN
        (
          (substring(e.question_id from 1 for length(e.question_id)-1))::int
          * (right(e.question_id, 1)::int)
        )::text
      ELSE NULL
    END
  ) AS answer,
  e.last_result,
  e.updated_at
FROM enriched e
LEFT JOIN topics t
  ON t.id = e.effective_topic_id
LEFT JOIN topics tp
  ON tp.id = t.parent_topic_id
LEFT JOIN subjects s
  ON s.id = t.subject_id
ORDER BY e.user_name, subject, topic_level1, topic_level2, e.updated_at DESC;
""")
wrong_ans_ques_df.head(2)

,user_name,subject,topic_level1,topic_level2,question_id,question,answer,last_result,updated_at
0,Bhavi,English,Word Spellings,3 Letter Words,104005,I got a good grade on the test.,grade,again,2026-04-10 20:59:03.873
1,Bhavi,Mathematics,Multiplication Tables,Tables 1-5,100033,4 x 3,12,again,2026-04-10 20:37:05.992


In [64]:
# Correct and wrong answer counts
correct_wrong_ans_counts_df = fetch_df("""
SELECT last_result, COUNT(*)
FROM reviews
GROUP BY last_result
ORDER BY COUNT(*) DESC;
""")
correct_wrong_ans_counts_df

,last_result,count
0,good,964
1,again,21
